# Name That LLM — offline inference

Attach your **trusted** bundle dataset containing `ntl_bundle.json`, `model.joblib`, and `wheels/`.
The bundle must be built for this Kaggle Python minor version and Linux platform.
Set Internet **off**. Verify the official competition input/output paths and ID dtype
before submitting; these defaults are provisional. No trained competition artifact
is included in this repository. A joblib file can execute code; use only your own artifacts.


In [ ]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

# Environment overrides support local integration checks without editing this notebook.
input_root = Path(os.environ.get("NTL_INPUT_ROOT", "/kaggle/input"))
found = sorted(input_root.rglob("ntl_bundle.json"))
if len(found) != 1:
    raise RuntimeError(f"Attach exactly one model bundle; found {len(found)} manifests")
bundle = found[0].parent
manifest = json.loads(found[0].read_text())
minor = f"{sys.version_info.major}.{sys.version_info.minor}"
if manifest["python_minor"] != minor:
    raise RuntimeError(f"Bundle Python {manifest['python_minor']} != runtime {minor}; rebuild wheels for this runtime")
for relative, expected in manifest["files"].items():
    path = (bundle / relative).resolve()
    if not path.is_relative_to(bundle.resolve()):
        raise RuntimeError("Unsafe bundle path")
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != expected:
        raise RuntimeError(f"Checksum mismatch: {relative}")
work = Path(os.environ.get("NTL_WORKING", "/kaggle/working"))
work.mkdir(parents=True, exist_ok=True)
site = work / "ntl_site"
if site.exists():
    raise RuntimeError("ntl_site already exists; use a fresh working directory for a clean offline run")
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                "--find-links", str(bundle / "wheels"), "--only-binary=:all:",
                "--no-compile", "--target", str(site), manifest["package"]], check=True)


In [ ]:
test_path = Path(os.environ.get("NTL_TEST_PATH", "/kaggle/input/competitions/name-that-llm/test.csv"))
output_path = work / "submission.parquet"
env = {**os.environ, "PYTHONPATH": str(site), "OMP_NUM_THREADS": "2", "OPENBLAS_NUM_THREADS": "2"}
# A fresh subprocess imports the attached package versions before any system ML packages.
subprocess.run([sys.executable, "-m", "ntl.cli", "predict",
                "--model", str(bundle / "model.joblib"), "--test", str(test_path),
                "--out", str(output_path), "--threads", "2",
                "--id-dtype", os.environ.get("NTL_ID_DTYPE", "string")], env=env, check=True)
print(f"Validated prediction file: {output_path}")
